
# PA1 — Normal‑Form Games: Utilities, Best Responses, and Equilibria
*Intelligent Agents — Programming Assignment 1*

**Goals**
- Represent 2‑player normal‑form games
- Compute expected utilities for mixed strategies
- Check best responses and Nash equilibrium (strict/weak)
- Detect strictly dominated strategies
- **Core**: Verify correlated equilibrium (CE) constraints for a given distribution
- **Extra Credit**: Compute a CE via a small LP (SciPy)

**What to submit:** your completed `.ipynb` on Canvas. Before submitting, do **Runtime → Restart and run all**.



## Setup
Run this cell first. You may use only standard Python/NumPy. For the extra‑credit CE solver you may import SciPy if needed.


In [1]:

import numpy as np

# Helper: pretty print small arrays
def show(arr, name=None):
    if name: print(name, "=", end=" ")
    print(np.array(arr))



## Game Representation
We represent a 2‑player game with two payoff matrices `U1` and `U2` of shape `(m, n)` where `m` is the number of player‑1 actions and `n` the number of player‑2 actions.
A mixed strategy is a probability vector of length `m` (or `n`).

You will implement utility calculations assuming row‑player is Player 1 and column‑player is Player 2.



### Task 1 — Expected Utility
Implement `expected_utility(U, p, q)` that returns the expected utility to the owner of payoff matrix `U` when Player 1 plays `p` and Player 2 plays `q`.

**Requirements**
- Validate shapes and probabilities (nonnegative, sum to 1 within a tolerance)
- Work for pure or mixed strategies
- Return a scalar `float`


In [2]:
# === implement expected_utility ===
def expected_utility(U, p, q, tol=1e-9):
    """Return E[u] for payoff matrix U (shape m x n) under mixed p (len m) vs q (len n).
    Validate that p, q are probability vectors within tolerance tol.
    """
    if 1-np.sum(p)>tol or 1-np.sum(q)>tol: raise ValueError
    u=0
    for i in range(len(p)):
      for j in range(len(q)):
        u+=p[i]*q[j]*U[i][j]
    return u





### Public Tests: Task 1


In [3]:

# Simple 2x2 test (Matching Pennies payoff for Player 1)
U1 = np.array([[1, -1],
               [-1, 1]], dtype=float)

p = np.array([0.5, 0.5])
q = np.array([0.5, 0.5])

# Expected utility should be 0 at uniform strategies
eu = expected_utility(U1, p, q)
assert abs(eu - 0.0) < 1e-9

# Pure vs pure
assert expected_utility(U1, np.array([1,0]), np.array([1,0])) == 1.0  # (row0, col0)
assert expected_utility(U1, np.array([1,0]), np.array([0,1])) == -1.0 # (row0, col1)
print("✅ Task 1 public tests passed")


✅ Task 1 public tests passed



### Task 2 — Best Response (pure BR to a mixed opponent)
Implement `is_best_response_pure(U1, U2, s_index, opp_mix, player)` that returns `True/False` indicating whether `s_index` (a *pure* action index) is a best response to the opponent’s mixed strategy `opp_mix` for the specified `player ∈ {1,2}`.

**Notes**
- Use `expected_utility` from Task 1.
- Ties count as best responses (weak BR). Later we’ll extend strictness options.


In [4]:

# === implement is_best_response_pure ===
def is_best_response_pure(U1, U2, s_index, opp_mix, player, tol=1e-9):
    """Return True if pure action s_index is a (weak) best response for 'player' against opp_mix.
    player = 1 or 2. Use U1 for player 1 utilities, U2 for player 2.
    """
    # expected_utility(U, p, q, tol=1e-9)
    if player==1:
      m = np.array([0]*U1.shape[0])
      move = m.copy()
      move[s_index] = 1
      u_s = expected_utility(U1, move, opp_mix, tol=tol)
    if player==2:
      m = np.array([0]*U2.shape[1])
      move = m.copy()
      move[s_index] = 1
      u_s = expected_utility(U2, opp_mix, move, tol=tol)
    for i in range(len(m)):
      if i==s_index: continue
      move = m.copy()
      move[i] = 1
      if player==1:
        if u_s<expected_utility(U1, move, opp_mix, tol=tol): return False
      if player==2:
        if u_s<expected_utility(U2, opp_mix, move, tol=tol): return False
    return True


### Public Tests: Task 2


In [5]:

# Prisoner's Dilemma
# Actions: 0 = Cooperate (C), 1 = Defect (D)
U1_PD = np.array([[-1, -3],
                  [ 0, -2]], dtype=float)
U2_PD = np.array([[-1,  0],
                  [-3, -2]], dtype=float)

# Against opponent mix leaning to C, best response is D for either player
opp = np.array([0.8, 0.2])
assert is_best_response_pure(U1_PD, U2_PD, s_index=1, opp_mix=opp, player=1)
assert is_best_response_pure(U1_PD, U2_PD, s_index=1, opp_mix=opp, player=2)
assert not is_best_response_pure(U1_PD, U2_PD, s_index=0, opp_mix=opp, player=1)
assert not is_best_response_pure(U1_PD, U2_PD, s_index=0, opp_mix=opp, player=2)
print("✅ Task 2 public tests passed")


✅ Task 2 public tests passed



### Task 3 — Nash Equilibrium Check (strict/weak)
Implement `is_nash_equilibrium(U1, U2, p, q, strict=False)` that returns `True/False` for whether `(p,q)` is a Nash equilibrium.

**Details**
- For **weak** NE (`strict=False`): every action in the support must be a best response (allowing ties).
- For **strict** NE (`strict=True`): support actions must be *strictly* better than any deviation.
- You may reuse `is_best_response_pure` and `expected_utility`.
- Assume finite action sets and that `p`, `q` have small supports (students can enumerate pure deviations).


In [6]:

# === implement is_nash_equilibrium ===
def is_nash_equilibrium(U1, U2, p, q, strict=False, tol=1e-9):
    """Return True iff (p,q) is a (weak or strict) Nash equilibrium in a 2-player normal-form game."""
    # is_best_response_pure(U1, U2, s_index, opp_mix, player, tol=1e-9)
    p1, p2 = (-1,-1)
    if strict:
      for i in range(len(p)):
        if p[i]<tol: continue
        if is_best_response_pure(U1, U2, i, q, 1, tol=tol):
          if p1>-1: return False
          p1 = i
      for i in range(len(q)):
        if q[i]<tol: continue
        if is_best_response_pure(U1, U2, i, p, 2, tol=tol):
          if p2>-1: return False
          p2 = i
      return p1>-1 and p2>-1

    for i in range(len(p)):
      if p[i]<tol: continue
      if not is_best_response_pure(U1, U2, i, q, 1, tol=tol): return False
    for i in range(len(q)):
      if q[i]<tol: continue
      if not is_best_response_pure(U1, U2, i, p, 2, tol=tol): return False
    return True



### Public Tests: Task 3


In [7]:

# Matching Pennies: (uniform, uniform) is a (weak) NE
U1 = np.array([[1, -1],
               [-1, 1]], dtype=float)
U2 = -U1.copy()
p = np.array([0.5, 0.5])
q = np.array([0.5, 0.5])
assert is_nash_equilibrium(U1, U2, p, q, strict=False)
assert not is_nash_equilibrium(U1, U2, p, q, strict=True)

U1 = np.array([[-1, -3],
                [ 0, -2]])
U2 = np.array([[-1,  0],
                [-3, -2]])
p_defect = np.array([0.0, 1.0])
q_defect = np.array([0.0, 1.0])
assert is_nash_equilibrium(U1, U2, p_defect, q_defect)
assert is_nash_equilibrium(U1, U2, p_defect, q_defect, strict=True)

p_coop = np.array([1.0, 0.0])
q_coop = np.array([1.0, 0.0])
assert not is_nash_equilibrium(U1, U2, p_coop, q_coop)
assert not is_nash_equilibrium(U1, U2, p_coop, q_coop, strict=True)

# Matching Pennies: (0.5, 0.5) mixed strategy is NE
U1_MP = np.array([[ 1, -1],
                  [-1,  1]])
U2_MP = -U1_MP

p_mix = np.array([0.5, 0.5])
q_mix = np.array([0.5, 0.5])
assert is_nash_equilibrium(U1_MP, U2_MP, p_mix, q_mix)
assert not is_nash_equilibrium(U1_MP, U2_MP, p_mix, q_mix, strict=True)

# Asymmetric game: pure NE at (row 1, col 0)
U1_asym = np.array([[1, 2],
                    [3, 0]])
U2_asym = np.array([[1, 3],
                    [2, 0]])
p = np.array([0.0, 1.0])
q = np.array([1.0, 0.0])
assert is_nash_equilibrium(U1_asym, U2_asym, p, q)
assert is_nash_equilibrium(U1_asym, U2_asym, p, q, strict=True)

# A non-NE strategy profile
p_bad = np.array([0.5, 0.5])
q_bad = np.array([1.0, 0.0])
assert not is_nash_equilibrium(U1_asym, U2_asym, p_bad, q_bad)
print("✅ Task 3 public tests passed")


✅ Task 3 public tests passed



### Task 4 — Strictly Dominated Strategies
Implement `strictly_dominated(U, player)` that returns a list (or boolean mask) of which pure strategies for `player` are **strictly dominated** (by possibly mixed opponents). For PA1 we’ll use a simple **pairwise strict dominance** check: a strategy `a` is strictly dominated if there exists another **pure** strategy `b` such that `u_i(b, a_{-i}) > u_i(a, a_{-i})` for **all** opponent pure actions.

*(This simplified definition ignores domination by mixed strategies to keep PA1 tractable.)*


In [8]:

# === implement strictly_dominated ===
def strictly_dominated(U, player):
    """Return indices (or mask) of strictly dominated pure strategies for `player`
    using pairwise strict dominance by another pure strategy.
    U is U1 if player==1 else U2.
    """
    mask = [False]*U.shape[player-1]
    for i in range(U.shape[player-1]): # i:player curr
      for j in range(U.shape[player-1]): # j:player other
        if i==j:continue
        flag = True
        for k in range(U.shape[abs(player-2)]): # k:opp
          if player == 1: u_i = U[i][k]; u_j = U[j][k]
          if player == 2: u_i = U[k][i]; u_j = U[k][j]
          if u_i >= u_j: flag = False; break # if U
        if flag: break
      mask[i] = flag
    # print(mask)
    return mask


### Public Tests: Task 4


In [9]:

# Simple dominance example:
# For player 1, action 0 is worse than action 1 in every column.
U1_dom = np.array([[0, 0, 0],
                   [1, 1, 1]], dtype=float)
dom = strictly_dominated(U1_dom, player=1)
# Expect that action 0 is dominated
assert (0 in dom) or (hasattr(dom, "__len__") and bool(dom[0]) and not bool(dom[1]))

U1_PD = np.array([[-1, -3],   # P1: C
                  [ 0, -2]])  # P1: D
U2_PD = np.array([[-1,  0],   # P2: C
                  [-3, -2]])  # P2: D

assert strictly_dominated(U1_PD, player=1) == [True, False]  # C is dominated
assert strictly_dominated(U2_PD, player=2) == [True, False]  # C is dominated

U1_MP = np.array([[ 1, -1],
                  [-1,  1]])
U2_MP = -U1_MP
assert strictly_dominated(U1_MP, player=1) == [False, False]
assert strictly_dominated(U2_MP, player=2) == [False, False]

U1 = np.array([[0, 0],     # Rock
               [1, 1]])    # Paper
assert strictly_dominated(U1, player=1) == [True, False]

U_sym = np.array([[1, 1],
                  [2, 2]])
assert strictly_dominated(U_sym, player=1) == [True, False]
assert strictly_dominated(U_sym, player=2) == [False, False]
print("✅ Task 4 public tests passed")


✅ Task 4 public tests passed



## Correlated Equilibrium (CE)
A joint distribution \(\sigma(a_1,a_2)\) is a **correlated equilibrium** if no player gains by deviating from the recommendation they receive.

### Task 5 (Core) — CE Verifier
Implement `is_correlated_equilibrium(U1, U2, sigma)` that returns `True/False` by checking the linear incentive constraints for both players:

\[
\sum_{a_{-i}} \sigma(a_i,a_{-i})\big(u_i(a_i,a_{-i}) - u_i(a_i',a_{-i})\big) \ge 0
\quad \text{for all } i, a_i, a_i'.
\]

Also verify \(\sigma\ge 0\) and \(\sum \sigma = 1\) within tolerance.


In [10]:

# === implement is_correlated_equilibrium (verifier) ===
def is_correlated_equilibrium(U1, U2, sigma, tol=1e-9):
    """Return True iff sigma (m x n) is a correlated equilibrium for (U1,U2)."""
    # expected_utility(U, p, q, tol=1e-9)
    if 1-np.sum(sigma)>tol: raise ValueError
    pure1 = np.zeros(U1.shape[0])
    pure2 = np.zeros(U1.shape[1])
    for index, s in np.ndenumerate(sigma):
      if s<tol: continue
      move1 = pure1.copy()
      move1[index[0]] = 1
      move2 = pure2.copy()
      move2[index[1]] = 1
      u_i = expected_utility(U1,move1,move2,tol)
      for i in range(U1.shape[0]):
        if i==index[0]: continue
        movealt = pure1.copy()
        movealt[i] = 1
        if u_i < expected_utility(U1,movealt,move2,tol): return False
      u_j = expected_utility(U2,move1,move2,tol)
      for j in range(U2.shape[1]):
        if j==index[1]: continue
        movealt = pure2.copy()
        movealt[j] = 1
        if u_j < expected_utility(U2,move1,movealt,tol): return False
    return True



### Public Tests: Task 5


In [11]:

# Coordination game: (2,2) for (0,0); (1,1) for (1,1)
U1_c = np.array([[2, 0],
                 [0, 1]], dtype=float)
U2_c = np.array([[2, 0],
                 [0, 1]], dtype=float)

# A trivial CE: recommend (0,0) with prob 1
sigma_trivial = np.array([[1.0, 0.0],
                          [0.0, 0.0]])
assert is_correlated_equilibrium(U1_c, U2_c, sigma_trivial)

# Uniform over the two coordinated outcomes is also a CE
sigma_two = np.array([[0.5, 0.0],
                       [0.0, 0.5]])
assert is_correlated_equilibrium(U1_c, U2_c, sigma_two)

sigma_four = np.array([[0.25, 0.25],
                       [0.25, 0.25]])
assert not is_correlated_equilibrium(U1_c, U2_c, sigma_four)

# Anti-coordination game: only off-diagonal outcomes good
U1_a = np.array([[0, 1],
                [1, 0]], dtype=float)
U2_a = np.array([[0, 1],
                [1, 0]], dtype=float)

assert is_correlated_equilibrium(U1_a, U2_a, np.array([[0.0, 0.5],
                                                       [0.5, 0.0]]))  # perfect coordination
assert not is_correlated_equilibrium(U1_a, U2_a, np.array([[0.5, 0.0],
                                                           [0.0, 0.5]]))  # both lose

# Prisoner's dilemma: only (D,D) is NE
U1_pd = np.array([[-1, -3],
                  [0,  -2]])
U2_pd = np.array([[-1,  0],
                  [-3, -2]])

assert is_correlated_equilibrium(U1_pd, U2_pd, np.array([[0.0, 0.0],
                                                          [0.0, 1.0]]))  # (D,D)
assert not is_correlated_equilibrium(U1_pd, U2_pd, np.array([[0.5, 0.0],
                                                             [0.0, 0.5]]))  # (C,C) and (D,D)

# Matching pennies: no pure NE, only mixed CE
U1_mp = np.array([[ 1, -1],
                  [-1,  1]], dtype=float)
U2_mp = -U1_mp

# The fully mixed CE
sigma_mix = np.array([[0.25, 0.25],
                      [0.25, 0.25]])
assert not is_correlated_equilibrium(U1_mp, U2_mp, sigma_mix)

## Asymmetric payoff: one-sided incentive
U1_asym = np.array([[2, 1],
                    [1, 0]])
U2_asym = np.array([[0, 1],
                    [1, 2]])
sigma_asym = np.array([[0.5, 0.0],
                       [0.0, 0.5]])
assert not is_correlated_equilibrium(U1_asym, U2_asym, sigma_asym)

print("✅ Task 5 public tests passed")


✅ Task 5 public tests passed



### Extra Credit — CE via LP (feasibility)
**Optional**: Implement `find_correlated_equilibrium(U1, U2)` that returns a feasible \(\sigma\) by solving a small LP (e.g., with SciPy `linprog`). Return `None` if no CE is found (should not occur for finite 2‑player games).

> If you implement this, include a small demo that verifies your \(\sigma\) with `is_correlated_equilibrium`.


In [12]:

# === OPTIONAL: implement find_correlated_equilibrium via LP ===
# from scipy.optimize import linprog  # if you choose to use SciPy
def find_correlated_equilibrium(U1, U2):
    """Optional: return a feasible sigma (m x n) that satisfies CE constraints, or None."""
    # TODO (extra credit)
    return None



## Methods Note (≈200–300 words)
Briefly discuss:
- How you validated probability vectors and handled tolerances
- How you handled strict vs. weak best responses/NE
- One limitation of pairwise strict dominance detection
- (If EC attempted) How you set up the LP and mapped constraints to matrices

I validate the probability vectors by ensuring 1-sum(vect) or 1-sum(mat) in correlated equilibrium is > tol. tolerance is only used for ensuring prob vectors sum to 1 and whether their values should be read as 0, tol is not used in any utility calculations or comparisons.
Weak NE means that every action must be a best action so we check every action for is_best_response_pure for both players
Strong NE means that there's a single action is a best action for each player so we iterate through them ensuring both players have a best action and that there are no ties
A limitation of pairwise strict dominance detection is that it won't detect mixed strategies that could be better than a pure one



## Final Checklist
- All **public tests** show ✅
- `Runtime → Restart and run all` completes without errors
- Notebook saved and submitted on Canvas as `.ipynb`
